In [1]:
import skimage.io
import skimage.transform
import os
import shutil
import random
import joblib
import constants
import time
import tensorflow as tf
import numpy as np
from joblib import Parallel, delayed
from sklearn.model_selection import train_test_split

PYTHONWARNINGS="ignore"

In [2]:
def img_to_512(dir_imgs, dir_labels, new_img_path, new_label_path):
    
    # Remove the directories so we can stat anew
    if os.path.isdir(new_img_path):
        shutil.rmtree(new_img_path)
    os.makedirs(new_img_path)
    if os.path.isdir(new_label_path):
        shutil.rmtree(new_label_path)
    os.makedirs(new_label_path)

    Parallel(n_jobs=-1, prefer='threads', verbose=10)(delayed(split_image_to_512)(filename, dir_imgs, dir_labels, new_img_path, new_label_path) for filename in os.listdir(os.path.join(os.getcwd(), dir_imgs)))
    # for filename in os.listdir(os.path.join(os.getcwd(), dir_imgs)):
    #     split_image_to_512()
    # print(f"{teller_files}/{aantal}")

def split_image_to_512(filename, dir_imgs, dir_labels, new_img_path, new_label_path):
    # print(filename)
    # print(f"{teller_files}/{aantal}", end='\r')
    filename = filename.split(".")[0]
    # Per image
    img = skimage.io.imread(f"{dir_imgs}/{filename}.JPG")
    labels = open(f"{dir_labels}/{filename}.txt")
    lines = labels.readlines()
    lines = [line.rstrip() for line in lines]
    teller = 1
    y = 0
    while(y < constants.Y_END - constants.WIDTH_PIECE): #zijkanten worden weggegooid indien te klein
        x = 0
        while(x < constants.X_END - constants.WIDTH_PIECE): #zijkanten worden weggegooid indien te klein
            x_end = x+constants.WIDTH_PIECE if x+constants.WIDTH_PIECE < constants.X_END else constants.X_END
            y_end = y+constants.WIDTH_PIECE if y+constants.WIDTH_PIECE < constants.Y_END else constants.Y_END
            img_piece = img[y:y_end,x:x_end]
            labels_piece = open(f"{new_label_path}/{filename}_{teller}.txt", "a")
            for line in lines:
                x_center, y_center, width, height = [round(float(s)) for s in line.split("\t")[1:]]
                if x_center >= x and x_center < x_end and y_center >= y and y_center < y_end:
                    labels_piece.write("0\t"+str(x_center-x)+"\t"+str(y_center-y)+"\t"+str(width)+"\t"+str(height)+"\n")
            labels_piece.close()
            if os.stat(labels_piece.name).st_size == 0: os.remove(labels_piece.name)
            skimage.io.imsave(f"{new_img_path}/{filename}_{teller}.JPG", img_piece)
            x += constants.WIDTH_PIECE - constants.OVERLAP #+550
            teller += 1
        y += constants.WIDTH_PIECE - constants.OVERLAP
    labels.close()

In [3]:
def split_clustering(path, sorted_path):
    kmeans = joblib.load('./trained_models/kmeans')

    # Remove the directories so we can stat anew
    if os.path.isdir(sorted_path):
        shutil.rmtree(sorted_path)
    os.makedirs(sorted_path)
    os.makedirs(f"{sorted_path}/1")
    os.makedirs(f"{sorted_path}/0")

    dataset = tf.keras.utils.image_dataset_from_directory(
        directory=path,
        labels=None,
        batch_size=10,
        image_size=(64,64),
        shuffle=False,
    )
    def process(image):
        image = image/255
        return image

    print("Start preprocessing")
    dataset = dataset.map(process)

    dataset = list(tf.data.Dataset.as_numpy_iterator(dataset.unbatch()))
    dataset = np.array(dataset)
    dataset = dataset.reshape((dataset.shape[0], -1))
    prediction = kmeans.predict(dataset)

    img_dir = os.listdir(path)

    print("Start filtering")
    Parallel(n_jobs=-1, verbose=2, prefer='threads')(
        delayed(filter)(filename, pred, path, sorted_path) for filename, pred in zip(img_dir, prediction)
    )

def filter(filename, sortdir, path, sorted_path):
    shutil.copy(f"{path}/{filename}", f"{sorted_path}/{sortdir}/{filename}")

In [4]:
def split_in_four_512(img_dir, label_dir, img_splitted, labels_splitted):
    # Remove the directory so we can stat anew
    if os.path.isdir(img_splitted):
        shutil.rmtree(img_splitted)
    os.makedirs(img_splitted)
    if os.path.isdir(labels_splitted):
        shutil.rmtree(labels_splitted)
    os.makedirs(labels_splitted)

    Parallel(n_jobs=-2, prefer='processes', verbose=1)(
        delayed(filter_split_512)(filename.split(".")[0], img_dir, label_dir, img_splitted, labels_splitted) for filename in os.listdir(img_dir)
    )

def filter_split_512(name_short, img_dir, label_dir, img_splitted, labels_splitted):        
    im = skimage.io.imread(f'{img_dir}/{name_short}.JPG')
    M = im.shape[0]//2
    N = im.shape[1]//2
    tiles = [im[x:x+M,y:y+N] for x in range(0,im.shape[0],M) for y in range(0,im.shape[1],N)]
    try:
        labels = open(f"{label_dir}/{name_short}.txt")
        lines = [line.rstrip() for line in labels.readlines()]
    except:
        lines = []
    for i in range(1, len(tiles)+1):
        skimage.io.imsave(f"{img_splitted}/{name_short}_{i}.JPG", tiles[i-1])
        labels_part = open(f"{labels_splitted}/{name_short}_{i}.txt", "a")
        for line in lines:
            name, x_center, y_center, width, height = [float(s) for s in line.split("\t")]
            if i == 1 and x_center < M and y_center < N:
                labels_part.write("0\t"+str(x_center)+"\t"+str(y_center)+"\t"+str(width)+"\t"+str(height)+"\n")
            elif i == 2 and x_center > M and y_center < N:
                labels_part.write("0\t"+str((x_center-M))+"\t"+str(y_center)+"\t"+str(width)+"\t"+str(height)+"\n")
            elif i == 3 and x_center < M and y_center > N:
                labels_part.write("0\t"+str(x_center)+"\t"+str((y_center-N))+"\t"+str(width)+"\t"+str(height)+"\n")
            elif i == 4 and x_center > M and y_center > N:
                labels_part.write("0\t"+str((x_center-M))+"\t"+str((y_center-N))+"\t"+str(width)+"\t"+str(height)+"\n")
        labels_part.close()
        if os.stat(labels_part.name).st_size == 0: os.remove(labels_part.name)

In [5]:
def split_in_four(img_dir, label_dir, img_splitted, labels_splitted):
    # Remove the directory so we can stat anew
    if os.path.isdir(img_splitted):
        shutil.rmtree(img_splitted)
    os.makedirs(img_splitted)
    if os.path.isdir(labels_splitted):
        shutil.rmtree(labels_splitted)
    os.makedirs(labels_splitted)

    img_list = os.listdir(img_dir)
    label_list = os.listdir(label_dir)

    Parallel(n_jobs=-2, prefer='processes', verbose=1)(
    delayed(filter_split)(filename.split(".")[0], img_dir, label_dir, img_splitted, labels_splitted, label_list) for filename in img_list
    )

def filter_split(name_short, img_dir, label_dir, img_splitted, labels_splitted, label_list):
        if f'{name_short}.txt' in label_list:
            im = skimage.io.imread(f'{img_dir}/{name_short}.JPG')
            M = im.shape[0]//2
            N = im.shape[1]//2
            tiles = [im[x:x+M,y:y+N] for x in range(0,im.shape[0],M) for y in range(0,im.shape[1],N)]
            try:
                labels = open(f"{label_dir}/{name_short}.txt")
                lines = [line.rstrip() for line in labels.readlines()]
            except:
                lines = []
            for i in range(1, len(tiles)+1):
                skimage.io.imsave(f"{img_splitted}/{name_short}_{i}.JPG", tiles[i-1])
                labels_part = open(f"{labels_splitted}/{name_short}_{i}.txt", "a")
                for line in lines:
                    name, x_center, y_center, width, height = [float(s) for s in line.split("\t")]
                    if i == 1 and x_center < M and y_center < N:
                        labels_part.write("0\t"+str(x_center)+"\t"+str(y_center)+"\t"+str(width)+"\t"+str(height)+"\n")
                    elif i == 2 and x_center > M and y_center < N:
                        labels_part.write("0\t"+str((x_center-M))+"\t"+str(y_center)+"\t"+str(width)+"\t"+str(height)+"\n")
                    elif i == 3 and x_center < M and y_center > N:
                        labels_part.write("0\t"+str(x_center)+"\t"+str((y_center-N))+"\t"+str(width)+"\t"+str(height)+"\n")
                    elif i == 4 and x_center > M and y_center > N:
                        labels_part.write("0\t"+str((x_center-M))+"\t"+str((y_center-N))+"\t"+str(width)+"\t"+str(height)+"\n")
                labels_part.close()
                if os.stat(labels_part.name).st_size == 0: os.remove(labels_part.name)
        

In [6]:
def prepare_model_data(dir_data, dir_labels, train_path, test_path):
    # Remove the directories so we can stat anew
    if os.path.isdir(train_path):
        shutil.rmtree(train_path)
    os.makedirs(train_path)
    os.makedirs(f"{train_path}/positive")
    os.makedirs(f"{train_path}/negative")
    if os.path.isdir(test_path):
        shutil.rmtree(test_path)
    os.makedirs(test_path)
    os.makedirs(f"{test_path}/positive")
    os.makedirs(f"{test_path}/negative")


    dir_list = os.listdir(dir_data)
    label_list = os.listdir(dir_labels)
    
    train_list, test_list = train_test_split(dir_list, train_size = 0.8)

    with Parallel(n_jobs=-2, prefer='processes', verbose=1) as parallel:
        parallel(delayed(filter_train_test)(filename, label_list, dir_data, train_path) for filename in train_list)
        parallel(delayed(filter_train_test)(filename, label_list, dir_data, test_path) for filename in test_list)

def filter_train_test(filename, label_list, dir_data, parentdir):
    labelname = filename.split(".")[0] + ".txt"
    if labelname in label_list:
        shutil.copy(f"{dir_data}/{filename}", f"{parentdir}/positive/{filename}")
    else:
        shutil.copy(f"{dir_data}/{filename}", f"{parentdir}/negative/{filename}")


In [3]:
# startTime = time.time()
img_to_512("./data/images_full/", "./data/labels_full_txt/",
           './data/images_512/', './data/label_512/')
# endTime = time.time()-startTime
# print(f'Time elapsed: {endTime//60}m {round(endTime%60,2)}s')

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 20 concurrent workers.
[Parallel(n_jobs=-1)]: Done   1 tasks      | elapsed:   21.5s
[Parallel(n_jobs=-1)]: Done  10 tasks      | elapsed:   22.2s
[Parallel(n_jobs=-1)]: Done  21 tasks      | elapsed:   43.1s
[Parallel(n_jobs=-1)]: Done  32 tasks      | elapsed:   43.9s
[Parallel(n_jobs=-1)]: Done  45 tasks      | elapsed:  1.1min
[Parallel(n_jobs=-1)]: Done  58 tasks      | elapsed:  1.1min
[Parallel(n_jobs=-1)]: Done  73 tasks      | elapsed:  1.4min
[Parallel(n_jobs=-1)]: Done  88 tasks      | elapsed:  1.8min
[Parallel(n_jobs=-1)]: Done 105 tasks      | elapsed:  2.2min
[Parallel(n_jobs=-1)]: Done 122 tasks      | elapsed:  2.5min
[Parallel(n_jobs=-1)]: Done 141 tasks      | elapsed:  2.9min
[Parallel(n_jobs=-1)]: Done 160 tasks      | elapsed:  3.1min
[Parallel(n_jobs=-1)]: Done 181 tasks      | elapsed:  3.7min
[Parallel(n_jobs=-1)]: Done 202 tasks      | elapsed:  4.1min
C:\Users\woutb\AppData\Local\Temp\ipykernel_16968\

In [5]:
startTime = time.time()
split_clustering('./data/images_512/', './data/images_512_kmeans/')
endTime = time.time()-startTime
print(f'Time elapsed: {endTime//60}m {round(endTime%60,2)}s')

Found 117450 files belonging to 1 classes.
Start preprocessing
Start filtering


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 20 concurrent workers.
[Parallel(n_jobs=-1)]: Done   1 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 122 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 325 tasks      | elapsed:    0.1s
[Parallel(n_jobs=-1)]: Done 608 tasks      | elapsed:    0.2s
[Parallel(n_jobs=-1)]: Done 973 tasks      | elapsed:    0.4s
[Parallel(n_jobs=-1)]: Done 1418 tasks      | elapsed:    0.7s
[Parallel(n_jobs=-1)]: Done 1945 tasks      | elapsed:    0.9s
[Parallel(n_jobs=-1)]: Done 2552 tasks      | elapsed:    1.2s
[Parallel(n_jobs=-1)]: Done 3241 tasks      | elapsed:    1.6s
[Parallel(n_jobs=-1)]: Done 4010 tasks      | elapsed:    2.2s
[Parallel(n_jobs=-1)]: Done 4861 tasks      | elapsed:    2.6s
[Parallel(n_jobs=-1)]: Done 5792 tasks      | elapsed:    3.2s
[Parallel(n_jobs=-1)]: Done 6805 tasks      | elapsed:    3.7s
[Parallel(n_jobs=-1)]: Done 7898 tasks      | elapsed:    4.3s
[Parallel(n_jobs=-1)]: Done 9073 tasks   

Time elapsed: 5.0m 42.79s


In [15]:
startTime = time.time()
split_in_four_512('./data/images_512_kmeans/0/', './data/label_512/',
              './data/images_256/', './data/label_256/')
endTime = time.time()-startTime
print(f'Time elapsed: {endTime//60}m {round(endTime%60,2)}s')

[Parallel(n_jobs=-2)]: Using backend LokyBackend with 19 concurrent workers.
[Parallel(n_jobs=-2)]: Done  12 tasks      | elapsed:    0.1s
[Parallel(n_jobs=-2)]: Done 286 tasks      | elapsed:    1.7s
[Parallel(n_jobs=-2)]: Done 786 tasks      | elapsed:    4.7s
[Parallel(n_jobs=-2)]: Done 1486 tasks      | elapsed:    8.9s
[Parallel(n_jobs=-2)]: Done 2386 tasks      | elapsed:   14.2s
[Parallel(n_jobs=-2)]: Done 3486 tasks      | elapsed:   20.7s
[Parallel(n_jobs=-2)]: Done 4786 tasks      | elapsed:   28.5s
[Parallel(n_jobs=-2)]: Done 6286 tasks      | elapsed:   37.5s
[Parallel(n_jobs=-2)]: Done 7986 tasks      | elapsed:   47.6s
[Parallel(n_jobs=-2)]: Done 9886 tasks      | elapsed:   59.0s
[Parallel(n_jobs=-2)]: Done 11986 tasks      | elapsed:  1.2min
[Parallel(n_jobs=-2)]: Done 14286 tasks      | elapsed:  1.4min
[Parallel(n_jobs=-2)]: Done 16786 tasks      | elapsed:  1.7min
[Parallel(n_jobs=-2)]: Done 19486 tasks      | elapsed:  2.0min
[Parallel(n_jobs=-2)]: Done 22386 tasks 

Time elapsed: 14.0m 47.6s


[Parallel(n_jobs=-2)]: Done 109608 out of 109608 | elapsed: 13.6min finished


In [12]:
startTime = time.time()
split_in_four('./data/images_256/', './data/label_256/',
              './data/images_128/', './data/label_128/')
endTime = time.time()-startTime
print(f'Time elapsed: {endTime//60}m {round(endTime%60,2)}s')

[Parallel(n_jobs=-2)]: Using backend LokyBackend with 19 concurrent workers.
[Parallel(n_jobs=-2)]: Done  12 tasks      | elapsed:    5.0s
[Parallel(n_jobs=-2)]: Done 191 tasks      | elapsed:    5.7s
[Parallel(n_jobs=-2)]: Done 4121 tasks      | elapsed:    7.5s
[Parallel(n_jobs=-2)]: Done 15993 tasks      | elapsed:   11.1s
[Parallel(n_jobs=-2)]: Done 61561 tasks      | elapsed:   15.7s
[Parallel(n_jobs=-2)]: Done 162937 tasks      | elapsed:   21.2s
[Parallel(n_jobs=-2)]: Done 329337 tasks      | elapsed:   29.5s


Time elapsed: 0.0m 42.38s


[Parallel(n_jobs=-2)]: Done 438432 out of 438432 | elapsed:   35.8s finished


In [13]:
startTime = time.time()
split_in_four('./data/images_128/', './data/label_128/',
              './data/images_64/', './data/label_64/')
endTime = time.time()-startTime
print(f'Time elapsed: {endTime//60}m {round(endTime%60,2)}s')

[Parallel(n_jobs=-2)]: Using backend LokyBackend with 19 concurrent workers.
[Parallel(n_jobs=-2)]: Done  12 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-2)]: Done 458 tasks      | elapsed:    0.9s
[Parallel(n_jobs=-2)]: Done 1510 tasks      | elapsed:    2.3s
[Parallel(n_jobs=-2)]: Done 6910 tasks      | elapsed:    8.7s
[Parallel(n_jobs=-2)]: Done 14110 tasks      | elapsed:   16.6s


Time elapsed: 0.0m 23.17s


[Parallel(n_jobs=-2)]: Done 16336 out of 16336 | elapsed:   19.2s finished


In [3]:
startTime = time.time()
prepare_model_data('./data/images_64/', './data/label_64/',
                   './data/images_64_train/', './data/images_64_test/')
endTime = time.time()-startTime
print(f'Time elapsed: {endTime//60}m {round(endTime%60,2)}s')

[Parallel(n_jobs=-2)]: Using backend LokyBackend with 19 concurrent workers.
[Parallel(n_jobs=-2)]: Done  12 tasks      | elapsed:    2.2s
[Parallel(n_jobs=-2)]: Done 230 tasks      | elapsed:    2.7s
[Parallel(n_jobs=-2)]: Done 2118 tasks      | elapsed:    5.6s
[Parallel(n_jobs=-2)]: Done 4918 tasks      | elapsed:    9.8s
[Parallel(n_jobs=-2)]: Done 8518 tasks      | elapsed:   14.6s
[Parallel(n_jobs=-2)]: Done 12918 tasks      | elapsed:   20.6s
[Parallel(n_jobs=-2)]: Done 14020 out of 14057 | elapsed:   22.2s remaining:    0.0s
[Parallel(n_jobs=-2)]: Done 14057 out of 14057 | elapsed:   22.2s finished
[Parallel(n_jobs=-2)]: Using backend LokyBackend with 19 concurrent workers.
[Parallel(n_jobs=-2)]: Done 134 tasks      | elapsed:    0.2s
[Parallel(n_jobs=-2)]: Done 1870 tasks      | elapsed:    2.7s
[Parallel(n_jobs=-2)]: Done 3395 tasks      | elapsed:    4.8s


Time elapsed: 0.0m 27.5s


[Parallel(n_jobs=-2)]: Done 3515 out of 3515 | elapsed:    5.0s finished


In [4]:
startTime = time.time()
prepare_model_data('./data/images_128/', './data/label_128/',
                   './data/images_128_train/', './data/images_128_test/')
endTime = time.time()-startTime
print(f'Time elapsed: {endTime//60}m {round(endTime%60,2)}s')

[Parallel(n_jobs=-2)]: Using backend LokyBackend with 19 concurrent workers.
[Parallel(n_jobs=-2)]: Done  12 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-2)]: Done 730 tasks      | elapsed:    1.0s
[Parallel(n_jobs=-2)]: Done 4730 tasks      | elapsed:    5.5s
[Parallel(n_jobs=-2)]: Done 10330 tasks      | elapsed:   11.7s
[Parallel(n_jobs=-2)]: Done 13068 out of 13068 | elapsed:   15.0s finished
[Parallel(n_jobs=-2)]: Using backend LokyBackend with 19 concurrent workers.
[Parallel(n_jobs=-2)]: Done 230 tasks      | elapsed:    0.4s
[Parallel(n_jobs=-2)]: Done 2630 tasks      | elapsed:    3.4s


Time elapsed: 0.0m 20.83s


[Parallel(n_jobs=-2)]: Done 3268 out of 3268 | elapsed:    4.4s finished


In [9]:
startTime = time.time()
prepare_model_data('./data/images_256/', './data/label_256/',
                   './data/images_256_train/', './data/images_256_test/')
endTime = time.time()-startTime
print(f'Time elapsed: {endTime//60}m {round(endTime%60,2)}s')

[Parallel(n_jobs=-2)]: Using backend LokyBackend with 19 concurrent workers.
[Parallel(n_jobs=-2)]: Done  12 tasks      | elapsed:    1.4s
[Parallel(n_jobs=-2)]: Done 287 tasks      | elapsed:    1.8s
[Parallel(n_jobs=-2)]: Done 6515 tasks      | elapsed:    4.1s
[Parallel(n_jobs=-2)]: Done 17715 tasks      | elapsed:    8.9s
[Parallel(n_jobs=-2)]: Done 32115 tasks      | elapsed:   15.8s
[Parallel(n_jobs=-2)]: Done 49715 tasks      | elapsed:   23.2s
[Parallel(n_jobs=-2)]: Done 70515 tasks      | elapsed:   33.9s
[Parallel(n_jobs=-2)]: Done 94515 tasks      | elapsed:   47.4s
[Parallel(n_jobs=-2)]: Done 121715 tasks      | elapsed:  1.1min
[Parallel(n_jobs=-2)]: Done 152115 tasks      | elapsed:  1.4min
[Parallel(n_jobs=-2)]: Done 185715 tasks      | elapsed:  1.7min
[Parallel(n_jobs=-2)]: Done 222515 tasks      | elapsed:  2.1min
[Parallel(n_jobs=-2)]: Done 262515 tasks      | elapsed:  2.4min
[Parallel(n_jobs=-2)]: Done 305715 tasks      | elapsed:  2.9min
[Parallel(n_jobs=-2)]: Don

Time elapsed: 4.0m 20.28s


[Parallel(n_jobs=-2)]: Done 87687 out of 87687 | elapsed:   45.7s finished


In [10]:
startTime = time.time()
path = './data/images_256_train/negative'
dir = os.listdir(path)
dir_list = random.sample(dir, len(dir)*2//3)
aantal = len(dir_list)
teller = 0
for filename in dir_list:
    teller += 1
    os.remove(f'{path}/{filename}')
    print(f'{teller}/{aantal}', end='\r')
print(f'{teller}/{aantal}')
endTime = time.time()-startTime
print(f'Time elapsed: {endTime//60}m {round(endTime%60,2)}s')

231647/231647
Time elapsed: 0.0m 41.08s


In [3]:
# path = './data/images_256_train/negative/'
# dir = os.listdir(path)
# part1 = dir[:len(dir)//4]
# part2 = dir[len(dir)//4:len(dir)//2]
# part3 = dir[len(dir)//2:len(dir)*3//4]
# part4 = dir[len(dir)*3//4:]

# for filename in part1:
#     shutil.copy(f'{path}/{filename}', f'./data/images_256_train_negative_1/{filename}')
# for filename in part2:
#     shutil.copy(f'{path}/{filename}', f'./data/images_256_train_negative_2/{filename}')
# for filename in part3:
#     shutil.copy(f'{path}/{filename}', f'./data/images_256_train_negative_3/{filename}')
# for filename in part4:
#     shutil.copy(f'{path}/{filename}', f'./data/images_256_train_negative_4/{filename}')

In [4]:
# path = './data/images_512'
# if os.path.isdir(f'{path}_1'):
#     shutil.rmtree(f'{path}_1')
# os.makedirs(f'{path}_1')
# if os.path.isdir(f'{path}_2'):
#     shutil.rmtree(f'{path}_2')
# os.makedirs(f'{path}_2')
# if os.path.isdir(f'{path}_3'):
#     shutil.rmtree(f'{path}_3')
# os.makedirs(f'{path}_3')
# if os.path.isdir(f'{path}_4'):
#     shutil.rmtree(f'{path}_4')
# os.makedirs(f'{path}_4')

# dir = os.listdir(path)
# part1 = dir[:len(dir)//4]
# part2 = dir[len(dir)//4:len(dir)//2]
# part3 = dir[len(dir)//2:len(dir)*3//4]
# part4 = dir[len(dir)*3//4:]

# for filename in part1:
#     shutil.copy(f'{path}/{filename}', f'{path}_1/{filename}')
# for filename in part2:
#     shutil.copy(f'{path}/{filename}', f'{path}_2/{filename}')
# for filename in part3:
#     shutil.copy(f'{path}/{filename}', f'{path}_3/{filename}')
# for filename in part4:
#     shutil.copy(f'{path}/{filename}', f'{path}_4/{filename}')